# Les 11 - Model Context Protocol (MCP)

Het **Model Context Protocol (MCP)** is een open standaard die agenten in staat stelt om dynamisch tools, bronnen en gegevensbronnen te ontdekken en te gebruiken tijdens runtime. In plaats van tools hard te coderen in een agent, laat MCP agenten verbinding maken met externe servers die mogelijkheden op aanvraag blootstellen.

In deze les leer je:
- Wat MCP is en waarom het belangrijk is voor agentensystemen
- Hoe de MCP client-serverarchitectuur werkt
- Hoe je agenten bouwt die toolontdekking in MCP-stijl gebruiken


## Installatie

**Vereisten:**
- Azure AI Foundry-project met een uitgerold model
- Voer `az login` uit voor authenticatie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Wat is het Model Context Protocol (MCP)?

MCP definieert een gestandaardiseerde manier voor AI-agenten om externe hulpmiddelen en gegevensbronnen te ontdekken en ermee te communiceren:

- **MCP Server**: Biedt tools, bronnen en prompts aan via een gestandaardiseerd protocol
- **MCP Client**: De agent-runtime die verbinding maakt met servers en beschikbare mogelijkheden ontdekt
- **Dynamische ontdekking**: Agents hebben geen hardgecodeerde tools nodig — ze ontdekken wat er beschikbaar is tijdens runtime

Dit is krachtig voor het bouwen van uitbreidbare agentsystemen waarbij nieuwe mogelijkheden kunnen worden toegevoegd zonder de agentcode te wijzigen.


## Hoe MCP werkt

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. De agent (MCP-client) maakt verbinding met een MCP-server
2. De server reageert met een lijst van beschikbare hulpmiddelen en hun schema's
3. De agent kan vervolgens tijdens het redeneren elk ontdekt hulpmiddel aanroepen
4. Resultaten stromen terug via hetzelfde protocol


## Simulatie van het ontdekken van MCP-tools

Aangezien een echte MCP-server een draaiend serverproces vereist, zullen we het patroon demonstreren met `@tool`-functies die simuleren wat een met MCP verbonden accommodatiedienst zou bieden.

In een productieomgeving zouden deze tools dynamisch door een MCP-server worden ontdekt in plaats van lokaal gedefinieerd.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Een agent bouwen met MCP-stijl hulpmiddelen


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP in productie

In een productieomgeving stelt MCP krachtige patronen mogelijk:

- **Dynamische tooldetectie**: Agenten verbinden met MCP-servers en ontdekken tools tijdens runtime
- **Ontkoppelde architectuur**: Toolproviders kunnen onafhankelijk van de agent updates doorvoeren
- **Organisatieoverstijgend delen**: Teams kunnen mogelijkheden via MCP-servers blootstellen die elke agent kan gebruiken
- **Ondersteuning voor Microsoft Agent Framework**: MAF heeft ingebouwde MCP-clientondersteuning via de `mcp`-integratie

Om een echte MCP-server met MAF te gebruiken, zou je verbinding maken via `hosted_mcp_tool()` of de MCP-clientintegratie.

**Meer informatie:**
- [MCP-specificatie](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP-ondersteuning](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Samenvatting

In deze les heb je geleerd:
- **MCP** is een open standaard voor dynamische ontdekking van tools tussen agenten en toolproviders
- De **client-serverarchitectuur** stelt agenten in staat om mogelijkheden tijdens runtime te ontdekken
- MCP maakt **uitbreidbare, ontkoppelde agentsystemen** mogelijk waarbij tools kunnen worden toegevoegd zonder codewijzigingen
- Microsoft Agent Framework biedt **ingebouwde MCP-ondersteuning** voor productiegebruik


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Disclaimer:
Dit document is vertaald met behulp van de AI-vertalingsdienst Co-op Translator (https://github.com/Azure/co-op-translator). Hoewel wij naar nauwkeurigheid streven, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het oorspronkelijke document in de oorspronkelijke taal moet als de gezaghebbende bron worden beschouwd. Voor cruciale informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
